# Day 24 – Feature Selection & Dimensionality Reduction
## From Too Many Variables to the Right Information

This notebook explores feature selection and PCA using a synthetic public-grievance dataset.

**Framework:** Understand → Select → Reduce → Validate → Interpret


## 1. Create a synthetic dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(24)
n = 500
volume = np.random.randint(700, 6000, n)
staff = np.random.randint(25, 120, n)
backlog = np.maximum(20, 0.18 * volume + np.random.normal(0, 180, n)).astype(int)
priority = np.random.uniform(0.05, 0.35, n)

df = pd.DataFrame({
    "complaint_volume": volume,
    "backlog": backlog,
    "staff_available": staff,
    "complaints_per_staff": volume / staff,
    "backlog_per_staff": backlog / staff,
    "priority_share": priority,
    "weekend_share": np.random.uniform(0.08, 0.30, n),
    "zone_pressure_index": 0.65 * volume / staff + 0.35 * backlog / staff + np.random.normal(0, 2, n),
    "irrelevant_noise": np.random.normal(0, 1, n)
})
df["avg_resolution_days"] = (
    2 + 0.0012 * volume + 0.0035 * backlog - 0.025 * staff
    + 7 * priority + np.random.normal(0, 1.7, n)
).clip(lower=1)
df.head()


## 2. Inspect correlations
Correlations can reveal redundant information, but do not prove predictive usefulness.

In [ ]:
X = df.drop(columns="avg_resolution_days")
y = df["avg_resolution_days"]
X.corr(numeric_only=True).round(2)


## 3. Filter method: variance threshold
This removes near-constant features. It does not use the target.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.01)
X_var = vt.fit_transform(X)
print("Original features:", X.shape[1])
print("After threshold:", len(vt.get_support(indices=True)))
print(X.columns[vt.get_support()].tolist())


## 4. Filter method: mutual information
Mutual information estimates statistical dependence between each feature and the target.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi = pd.Series(
    mutual_info_regression(X, y, random_state=24),
    index=X.columns
).sort_values(ascending=False)
mi


In [ ]:
mi.sort_values().plot(kind="barh", figsize=(8, 5), title="Mutual Information Scores")
plt.xlabel("Estimated mutual information")
plt.tight_layout()
plt.show()


## 5. Embedded method: Lasso
Lasso uses L1 regularization and can shrink some coefficients to zero. Correlated predictors can make selection unstable.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=24, max_iter=20000))
lasso.fit(X, y)
pd.Series(lasso.named_steps["lassocv"].coef_, index=X.columns).sort_values(key=np.abs, ascending=False)


## 6. Wrapper method: Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

X_scaled = StandardScaler().fit_transform(X)
rfe = RFE(LinearRegression(), n_features_to_select=5).fit(X_scaled, y)
pd.DataFrame({"feature": X.columns, "selected": rfe.support_, "rank": rfe.ranking_}).sort_values("rank")


## 7. Dimensionality reduction with PCA
Standardize numeric predictors before PCA. In predictive workflows, fit preprocessing on training data only.

In [ ]:
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_std = scaler.fit_transform(X)
pca = PCA().fit(X_std)
explained = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(X.shape[1])])
explained


In [ ]:
explained.cumsum().plot(marker="o", figsize=(8, 5), title="PCA – Cumulative Explained Variance")
plt.axhline(0.90, linestyle="--", label="90%")
plt.xlabel("Principal component")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Visualize two principal components

In [ ]:
pca2 = PCA(n_components=2)
coords = pca2.fit_transform(X_std)
plt.figure(figsize=(8, 5))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=y, cmap="viridis", alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Records in PCA Space")
plt.colorbar(sc, label="Average resolution days")
plt.tight_layout()
plt.show()
print("Variance explained by first two components:", round(pca2.explained_variance_ratio_.sum(), 3))


## 9. Compare held-out model performance
Preprocessing and selection are fitted within pipelines to avoid test-set leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

models = {
    "All features": Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]),
    "Top 5 MI features": Pipeline([
        ("select", SelectKBest(score_func=mutual_info_regression, k=5)),
        ("scale", StandardScaler()), ("model", LinearRegression())
    ]),
    "PCA (90% variance)": Pipeline([
        ("scale", StandardScaler()), ("pca", PCA(n_components=0.90)),
        ("model", LinearRegression())
    ])
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })
pd.DataFrame(rows).set_index("Model").round(3)


## 10. Practical checklist

- Define the prediction objective.
- Remove identifiers and invalid predictors.
- Inspect missingness, variance, and redundancy.
- Compare filter, wrapper, and embedded methods as appropriate.
- Fit selection and reduction steps on training data only.
- Use cross-validation for model selection.
- Evaluate on held-out data.
- Check stability, interpretability, and operational usefulness.
- Document why each feature or component is retained.

## Key Takeaway

**Feature selection retains a subset of original variables. Dimensionality reduction creates a smaller representation of the data.**

Neither guarantees better predictions; validate against the actual objective.

**Next:** Day 25 – Clustering & Unsupervised Learning.
